# 🤖 Mini Recruitment Chatbot — Advanced Prompt Engineering with LangChain

**Scenario:** I work for a recruitment agency and I'm building a chatbot assistant that:

* talks to job candidates in a **friendly, concise tone** (role + few-shot prompting),
* **extracts structured candidate information** (name, desired role, years of experience) using **function / tool calling**,
* returns **JSON** that could be sent straight to an HR system,
* **remembers the conversation** using a memory buffer, so candidates never have to repeat themselves.

**Stack:** Python · LangChain (`langchain-core`) · Google Gemini (`langchain-google-genai`) · Pydantic · Gradio (optional UI)

**How to run:** get a free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey), then use *Runtime → Run all*. You'll be asked for the key in Section 3.

## 1. Install Libraries

Only two packages are needed for the core chatbot:

* `langchain-core` – prompts, messages, tools and runnables (the LangChain building blocks)
* `langchain-google-genai` – the LangChain integration for Gemini chat models

`pydantic` is already installed as a dependency of LangChain. Gradio is installed separately in the optional Section 10, so the core notebook doesn't depend on it.

In [1]:
!pip install -qU langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.0 which is incompatible.


## 2. Imports

In [2]:
# Standard library
import json
import os
import re
from getpass import getpass
from typing import Optional

# Data validation (used for the candidate schema)
from pydantic import BaseModel, Field, ValidationError, field_validator

# LangChain building blocks
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

# Gemini chat model integration
from langchain_google_genai import ChatGoogleGenerativeAI

## 3. API Key Setup & Connect to the LLM

The API key is **never hard-coded**. The cell first checks Colab's *Secrets* panel (🔑 icon on the left, secret name `GOOGLE_API_KEY`) and otherwise asks for the key with `getpass`, so it isn't shown on screen or saved in the notebook.

I'm using **Gemini** through `ChatGoogleGenerativeAI`. Flash models are fast, cheap and available on the free tier. If the model name below isn't available for your key, swap it for `"gemini-flash-latest"` or another model listed in AI Studio.

**Handling a busy API:** Gemini sometimes returns `503 UNAVAILABLE` ("model is experiencing high demand") or `429` rate-limit errors. Two things protect against that:

* `max_retries=6`: the client automatically retries these errors with increasing waits. (In this library `max_retries` counts the *total* attempts, so `1` means no retries at all.)
* A **backup model**: `with_fallbacks(...)` sends the request to a second, lighter Gemini model if the main one still fails.

In [3]:
# --- API key (kept out of the code) ---
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata  # Colab "Secrets" panel
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [4]:
# --- Connect to the LLM ---
MODEL_NAME = "gemini-3.5-flash"                # main model (alternatives: "gemini-flash-latest", "gemini-2.5-flash")
BACKUP_MODEL_NAME = "gemini-flash-lite-latest"  # used only if the main model is overloaded / unavailable

# max_retries = total attempts per request; 503 and 429 errors are retried with backoff
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, max_retries=6)
backup_llm = ChatGoogleGenerativeAI(model=BACKUP_MODEL_NAME, max_retries=3)

# Quick connection check (falls back to the backup model if the main one is busy)
print(llm.with_fallbacks([backup_llm]).invoke("Reply with exactly: Connected to Gemini!").text)

Connected to Gemini!


## 4. Define the Recruitment Assistant Prompt

This is where most of the prompt engineering happens. The system prompt combines three techniques:

1. **Role prompting** – the model is told exactly who it is: *"a friendly recruitment assistant helping job seekers with career questions and extracting their basic information for job applications."*
2. **Explicit rules** – tone (friendly, concise), formatting (short intro + bullet points, no JSON for normal questions), memory behaviour (don't ask for details twice) and extraction rules (call the tool, never invent data, omit unknown experience instead of using `0`).
3. **Few-shot examples** – four short examples showing the expected tone for career advice and *when* to call the extraction tool (including a two-turn example where details are combined across messages).

**Design decision:** I put the few-shot examples *inside the system prompt* (clearly labelled as examples) rather than as fake chat messages. If they were real `HumanMessage`/`AIMessage` pairs in the history, the model could confuse the example candidate "Sam" with the real person it's talking to.

The final prompt template is: **system prompt → conversation history (memory) → new user message**.

In [5]:
SYSTEM_PROMPT = """You are a friendly recruitment assistant helping job seekers with career questions and extracting their basic information for job applications. You work for a recruitment agency.

## Tone and format for career questions
- Be friendly, encouraging and professional, but concise (no more than about 120 words).
- Give practical, actionable advice: one short opening sentence, then 3-5 bullet points.
- Stay focused on careers (job search, CVs, interviews, skills, career changes). If a question is unrelated, politely steer the conversation back to careers.
- Answer normal questions in plain text. Never answer them with JSON.

## Candidate information and the extract_candidate_info tool
- Candidate information means: the candidate's name, the role they want, and their years of experience.
- Remember everything the candidate has already told you in this conversation. Never ask again for a detail they already gave.
- As soon as you know BOTH the candidate's name AND their desired role (from the current message or from earlier messages), call the extract_candidate_info tool. Combine details from different messages.
- If you only know part of it (for example just the name), do NOT call the tool. Reply warmly and ask only for the missing detail.
- If the candidate corrects or adds a detail later, call the tool again with the complete, updated details.
- Include experience_years only if the candidate explicitly states it. If it was not mentioned, leave the field out completely. Never guess it and never use 0 as a placeholder.
- Never invent, guess or assume any detail.
- Write desired_role as a clean job title in Title Case, without words like "role" or "position" (for example "a developer role" -> "Developer").
- Convert written numbers to integers (for example "three years" -> 3).
- Always use the real tool call for extraction. Never write the tool call or the JSON as plain text.

## Examples (to show style only; they are NOT part of the real conversation)

Example 1
User: How do I prepare for a data analyst interview?
Assistant: Great choice of role! Focus on these areas:
- SQL: practise joins, GROUP BY and window functions.
- Excel or Python: be ready to clean and summarise a small dataset.
- Statistics: review averages, distributions and A/B testing basics.
- Storytelling: prepare 2-3 projects you can explain with the STAR method.
Good luck, you've got this!

Example 2
User: What skills should I learn to become a software developer?
Assistant: Here's a practical starting path:
- Learn one language well (Python or JavaScript are great first picks).
- Use Git and GitHub for all your projects.
- Practise data structures and problem solving.
- Build 2-3 small projects you can show employers.
Consistency beats cramming, so a little every day goes a long way!

Example 3
User: My name is Sam, I am applying for a developer role and I have 3 years of experience.
Assistant action: call extract_candidate_info with name="Sam", desired_role="Developer", experience_years=3

Example 4 (details spread over two messages, experience not given)
User: My name is Priya.
Assistant: Nice to meet you, Priya! What role are you interested in?
User: I'd like a UX designer position.
Assistant action: call extract_candidate_info with name="Priya", desired_role="UX Designer" (experience_years is left out because it was never mentioned)
"""

# system prompt -> memory buffer -> new user message
# (SystemMessage is used directly so the prompt text is not treated as a template,
#  which means literal braces in it would never break formatting.)
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

print(f"System prompt length: {len(SYSTEM_PROMPT.split())} words")
print("Prompt input variables:", prompt.input_variables)

System prompt length: 542 words
Prompt input variables: ['history', 'input']


## 5. Define Candidate Information Schema

The assignment gives the function schema as JSON (shown below for reference). I implement the same structure as a **Pydantic model**, which LangChain converts into a tool schema automatically. Pydantic also gives me validation for free:

| Field | Type | Required | Notes |
|---|---|---|---|
| `name` | string | ✅ | whitespace trimmed |
| `desired_role` | string | ✅ | normalised to Title Case |
| `experience_years` | integer | ❌ | `None` when not provided (never `0` by default), must be 0–60 |

`experience_years` is `Optional[int] = None`, so when a candidate doesn't mention experience the field is simply **left out** of the JSON output.

In [6]:
# The schema from the assignment brief (kept for reference / comparison)
EXTRACT_CANDIDATE_INFO_SCHEMA = {
    "name": "extract_candidate_info",
    "description": "Extract candidate information for job applications",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "desired_role": {"type": "string"},
            "experience_years": {"type": "integer"},
        },
        "required": ["name", "desired_role"],
    },
}


class CandidateInfo(BaseModel):
    """Extract candidate information for job applications."""

    name: str = Field(
        min_length=1,
        description="The candidate's name exactly as they stated it, e.g. 'Sam'.",
    )
    desired_role: str = Field(
        min_length=1,
        description="The job title the candidate is applying for, in Title Case, e.g. 'Data Analyst'.",
    )
    experience_years: Optional[int] = Field(
        default=None,
        ge=0,
        le=60,
        description=(
            "Total years of professional experience as a whole number. "
            "Only set this if the candidate explicitly stated it; otherwise leave it out. Never guess."
        ),
    )

    @field_validator("name")
    @classmethod
    def clean_name(cls, value: str) -> str:
        return value.strip()

    @field_validator("desired_role")
    @classmethod
    def title_case_role(cls, value: str) -> str:
        # Capitalise the first letter of each word but keep acronyms like "UX" or "QA" intact
        return " ".join(word[:1].upper() + word[1:] for word in value.split())


# Quick validation checks (no LLM needed)
print(CandidateInfo(name=" Sam ", desired_role="developer", experience_years=3).model_dump(exclude_none=True))
print(CandidateInfo(name="Sarah", desired_role="marketing analyst").model_dump(exclude_none=True))

{'name': 'Sam', 'desired_role': 'Developer', 'experience_years': 3}
{'name': 'Sarah', 'desired_role': 'Marketing Analyst'}


## 6. Implement Function / Tool Calling

The `@tool` decorator turns a Python function into a LangChain tool named **`extract_candidate_info`**, using `CandidateInfo` as its argument schema. Then `llm.bind_tools([...])` sends the tool definition to Gemini with every request.

How function calling works here:

1. The model reads the conversation and **decides** whether the user has given candidate details.
2. If yes, instead of writing text it returns a **tool call**: the tool name plus arguments that match the schema (available on `response.tool_calls`).
3. My code runs the tool, which validates the arguments with Pydantic and returns a clean Python dict → JSON.

The cell below also checks that the generated tool schema matches the one from the assignment brief.

In [7]:
@tool("extract_candidate_info", args_schema=CandidateInfo)
def extract_candidate_info(name: str, desired_role: str, experience_years: Optional[int] = None) -> dict:
    """Extract candidate information for job applications."""
    candidate = CandidateInfo(name=name, desired_role=desired_role, experience_years=experience_years)
    # exclude_none=True -> experience_years is omitted (not 0) when it wasn't provided
    return candidate.model_dump(exclude_none=True)


# Give the model access to the tool (the backup model gets the same tool, in case Gemini is overloaded)
llm_with_tools = llm.bind_tools([extract_candidate_info]).with_fallbacks(
    [backup_llm.bind_tools([extract_candidate_info])]
)

# --- Check: generated schema vs. the assignment schema ---
generated = convert_to_openai_tool(extract_candidate_info)["function"]
print("Tool name:        ", generated["name"])
print("Description:      ", generated["description"])
print("Properties match: ", set(generated["parameters"]["properties"]) == set(EXTRACT_CANDIDATE_INFO_SCHEMA["parameters"]["properties"]))
print("Required match:   ", generated["parameters"]["required"] == EXTRACT_CANDIDATE_INFO_SCHEMA["parameters"]["required"])

# --- Check: run the tool directly (no LLM) ---
print("\nDirect tool call:", extract_candidate_info.invoke({"name": "Sam", "desired_role": "developer", "experience_years": 3}))

# --- Check: let the model decide to call the tool ---
raw = llm_with_tools.invoke("My name is Sam and I am applying for a software developer role with 3 years of experience.")
print("\nRaw tool calls returned by the model:")
print(raw.tool_calls)

Tool name:         extract_candidate_info
Description:       Extract candidate information for job applications.
Properties match:  True
Required match:    True

Direct tool call: {'name': 'Sam', 'desired_role': 'Developer', 'experience_years': 3}

Raw tool calls returned by the model:
[{'name': 'extract_candidate_info', 'args': {'desired_role': 'Software Developer', 'name': 'Sam', 'experience_years': 3}, 'id': 'call_76666', 'type': 'tool_call'}]


## 7. Add Conversational Memory

The assignment asks for a **memory buffer**. In older LangChain this was `ConversationBufferMemory`. In current LangChain (1.x) that class has been removed, and `RunnableWithMessageHistory` / `InMemoryChatMessageHistory` are deprecated too (the docs now point to LangGraph persistence, which felt like overkill for a single chatbot).

So I implemented the same behaviour directly: a **buffer of LangChain message objects** (`HumanMessage` / `AIMessage`) for each `session_id`. Every turn is appended to the buffer, and the whole buffer is inserted into the prompt through `MessagesPlaceholder("history")`. That's exactly what `ConversationBufferMemory` did: keep the full conversation and send it back to the model each time.

Separate `session_id`s keep different candidates' conversations apart (e.g. Sam's test won't leak into Sarah's).

In [8]:
# session_id -> list of messages (the conversation buffer)
memory_store: dict[str, list[BaseMessage]] = {}


def get_memory(session_id: str) -> list[BaseMessage]:
    """Return the message buffer for a session, creating it if needed."""
    return memory_store.setdefault(session_id, [])


def save_to_memory(session_id: str, user_message: str, assistant_message: str) -> None:
    """Append one user/assistant exchange to the buffer."""
    get_memory(session_id).extend([
        HumanMessage(content=user_message),
        AIMessage(content=assistant_message),
    ])


def clear_memory(session_id: str) -> None:
    """Forget a conversation."""
    memory_store.pop(session_id, None)


def show_memory(session_id: str) -> None:
    """Print the buffer, handy for checking what the bot remembers."""
    print(f"--- Memory for session '{session_id}' ---")
    for msg in get_memory(session_id):
        speaker = "User" if isinstance(msg, HumanMessage) else "Assistant"
        print(f"{speaker}: {msg.content}")


# Tiny demo of the buffer (no LLM call)
save_to_memory("demo", "Hi there!", "Hello! How can I help with your career today?")
show_memory("demo")
clear_memory("demo")

--- Memory for session 'demo' ---
User: Hi there!
Assistant: Hello! How can I help with your career today?


## 8. Build the Chatbot

`chatbot_response(message, session_id)` ties everything together:

1. Load the conversation **history** for the session.
2. Build the prompt (**system prompt + history + new message**) and send it to the tool-enabled model (`prompt | llm_with_tools`).
3. **Detect tool calls**: if the model called `extract_candidate_info`, run the tool and validate the arguments.
4. Return either **clean JSON** (`json.dumps(..., indent=2)`) for extractions or **normal text** for career questions.
5. **Update memory** with the exchange.

A few safety nets handle problems the prompt alone can't guarantee:

* **Invalid tool arguments** (e.g. a missing role) are caught as `ValidationError` and the bot asks a follow-up question instead of crashing.
* **JSON written as text:** if the model writes JSON in its reply instead of making a real tool call, it's parsed and validated with the same schema.
* **Invented experience:** if the model fills in `experience_years` but the candidate never mentioned years or experience, the value is dropped. This enforces the "never invent data" rule in code.
* **API errors:** the request is retried and then sent to the backup model (Section 3). If both fail, the bot returns a friendly message and nothing is saved to memory, so the conversation can simply continue.

For extraction turns, the memory stores a short plain-language note (not raw JSON), so the model remembers what was recorded without starting to copy JSON into normal answers.

In [9]:
# The chain: prompt template -> Gemini (with the tool bound)
chatbot_chain = prompt | llm_with_tools


def run_extraction_tool(tool_calls: list[dict]) -> Optional[dict]:
    """Execute the extract_candidate_info tool call, if present. Returns None if args are invalid."""
    for call in tool_calls:
        if call["name"] == "extract_candidate_info":
            try:
                return extract_candidate_info.invoke(call["args"])
            except ValidationError as err:
                print(f"[debug] Tool arguments failed validation: {err.errors()}")
    return None


def parse_json_fallback(text: str) -> Optional[dict]:
    """Safety net: accept candidate JSON that the model wrote as plain text instead of a tool call."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return CandidateInfo.model_validate(json.loads(match.group())).model_dump(exclude_none=True)
    except (json.JSONDecodeError, ValidationError):
        return None


def drop_unstated_experience(candidate: dict, user_messages: list[str]) -> dict:
    """Remove experience_years if the candidate never talked about years/experience (prevents invented values)."""
    all_user_text = " ".join(user_messages).lower()
    if "experience_years" in candidate and not re.search(r"\b(years?|yrs?|experience|experienced)\b", all_user_text):
        candidate = {k: v for k, v in candidate.items() if k != "experience_years"}
    return candidate


def to_text(ai_message: AIMessage) -> str:
    """Get plain text from a model reply (Gemini can return a list of content blocks)."""
    return (ai_message.text or "").strip()


def chatbot_response(message: str, session_id: str = "default") -> str:
    """Send one user message to the chatbot and return either text or a JSON string."""
    history = get_memory(session_id)

    # 1-2. Send system prompt + history + new message to the model
    try:
        ai_message = chatbot_chain.invoke({"history": history, "input": message})
    except Exception as err:  # network / quota / auth problems
        return f"Sorry, I couldn't reach the language model right now ({type(err).__name__}: {err})."

    # 3. Did the model call the extraction tool? (or write JSON as text as a fallback)
    reply_text = to_text(ai_message)
    candidate = run_extraction_tool(ai_message.tool_calls) if ai_message.tool_calls else parse_json_fallback(reply_text)

    if candidate:
        previous_user_messages = [m.content for m in history if isinstance(m, HumanMessage)]
        candidate = drop_unstated_experience(candidate, previous_user_messages + [message])
        output = json.dumps(candidate, indent=2)

        # 5. Store a short note so the model remembers what it already recorded
        details = ", ".join(f"{key}={value}" for key, value in candidate.items())
        save_to_memory(session_id, message, f"(I recorded the candidate's details with extract_candidate_info: {details}.)")
        return output

    # 4. Normal conversational answer
    if not reply_text:  # e.g. a tool call with invalid arguments
        reply_text = "Thanks! Could you tell me your name and the role you're applying for?"
    save_to_memory(session_id, message, reply_text)
    return reply_text


def chat(message: str, session_id: str = "default") -> str:
    """Convenience wrapper: prints the exchange and returns the response."""
    response = chatbot_response(message, session_id)
    print(f"🧑 User: {message}\n\n🤖 Assistant:\n{response}\n")
    return response

## 9. Test the Chatbot

The assignment requires at least **5 tests**: **3 conversational Q&As** and **2 structured extractions**. I also added a **memory test**, where the candidate's details are spread across several messages (plus a correction at the end).

For each test the helper prints the question and the response, then **verifies** the result:

* Conversational tests: the answer must be normal text, **not** JSON.
* Extraction tests: the output must be **valid JSON** that matches the expected dictionary exactly. For Sarah, that means there must be **no** `experience_years` key, because she never mentioned it.

Each extraction test uses its own `session_id`, so one candidate's details can't leak into another test.

In [10]:
test_results = []


def is_json(text: str) -> bool:
    try:
        json.loads(text)
        return True
    except (json.JSONDecodeError, TypeError):
        return False


def normalise(data: dict) -> dict:
    """Case-insensitive comparison for string values."""
    return {k: v.lower() if isinstance(v, str) else v for k, v in data.items()}


def run_test(title: str, message: str, session_id: str, expected: Optional[dict] = None) -> str:
    print("=" * 80)
    print(title)
    print("=" * 80)
    response = chat(message, session_id)

    if expected is None:  # conversational test
        passed = not is_json(response) and not response.startswith("Sorry, I couldn't reach")
        print("✅ PASS: conversational text response" if passed else "❌ FAIL: expected conversational text")
    else:  # extraction test
        passed = is_json(response) and normalise(json.loads(response)) == normalise(expected)
        print("✅ PASS: valid JSON matching the expected output" if passed
              else f"❌ FAIL: expected\n{json.dumps(expected, indent=2)}")
    test_results.append((title, passed))
    print()
    return response

### Tests 1–3: Conversational Q&A

In [11]:
run_test("TEST 1 – Conversational", "How do I prepare for a data analyst interview?", session_id="career-qa")
run_test("TEST 2 – Conversational", "What skills should I improve for a software engineering job?", session_id="career-qa")
run_test("TEST 3 – Conversational", "How can I make my CV stronger?", session_id="career-qa");

TEST 1 – Conversational
🧑 User: How do I prepare for a data analyst interview?

🤖 Assistant:
Preparing for a data analyst interview is all about demonstrating your analytical mindset and technical skills! Focus on these key areas:

* **SQL & Tools**: Practice writing queries using joins, aggregations, and window functions, as these are frequently tested.
* **Data Visualization**: Be ready to explain how you design clear, actionable dashboards using tools like Tableau or Power BI.
* **Business Sense**: Research the company and think about how they use data to make business decisions.
* **The STAR Method**: Prepare 2-3 structured stories about past projects where you solved a real-world problem using data.

With some focused preparation, you'll do great!

✅ PASS: conversational text response

TEST 2 – Conversational
🧑 User: What skills should I improve for a software engineering job?

🤖 Assistant:
Upgrading your skills is a great way to stand out in the software engineering job market! F

### Tests 4–5: Structured extraction (function calling → JSON)

In [12]:
run_test(
    "TEST 4 – Extraction (with experience)",
    "My name is Sam and I am applying for a developer role with 3 years of experience.",
    session_id="candidate-sam",
    expected={"name": "Sam", "desired_role": "Developer", "experience_years": 3},
)

run_test(
    "TEST 5 – Extraction (no experience given -> field omitted, not 0)",
    "My name is Sarah and I am applying for a marketing analyst role.",
    session_id="candidate-sarah",
    expected={"name": "Sarah", "desired_role": "Marketing Analyst"},
);

TEST 4 – Extraction (with experience)
🧑 User: My name is Sam and I am applying for a developer role with 3 years of experience.

🤖 Assistant:
{
  "name": "Sam",
  "desired_role": "Developer",
  "experience_years": 3
}

✅ PASS: valid JSON matching the expected output

TEST 5 – Extraction (no experience given -> field omitted, not 0)
🧑 User: My name is Sarah and I am applying for a marketing analyst role.

🤖 Assistant:
{
  "name": "Sarah",
  "desired_role": "Marketing Analyst"
}

✅ PASS: valid JSON matching the expected output



### Test 6: Memory demonstration

John gives his name first and his role/experience in a later message. The bot should **not** ask for his name again, and it should combine both turns into one JSON record. Then he corrects his experience, and the bot should update the record while still remembering his name and role.

In [13]:
clear_memory("candidate-john")

run_test("TEST 6a – Memory (partial info: name only, no JSON expected yet)",
         "My name is John.", session_id="candidate-john")

run_test("TEST 6b – Memory (combine with previous turn)",
         "I am applying for a business analyst role and I have 2 years of experience.",
         session_id="candidate-john",
         expected={"name": "John", "desired_role": "Business Analyst", "experience_years": 2})

run_test("TEST 6c – Memory (correction: update only the experience)",
         "Sorry, correction: I actually have 3 years of experience.",
         session_id="candidate-john",
         expected={"name": "John", "desired_role": "Business Analyst", "experience_years": 3})

show_memory("candidate-john")

TEST 6a – Memory (partial info: name only, no JSON expected yet)
🧑 User: My name is John.

🤖 Assistant:
Sorry, I couldn't reach the language model right now (GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}).

❌ FAIL: expected conversational text

TEST 6b – Memory (combine with previous turn)
🧑 User: I am applying for a business analyst role and I have 2 years of experience.

🤖 Assistant:
Sorry, I couldn't reach the language model right now (GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}).

❌ FAIL: expected
{
  "name": "John",
  "desired_role": "Business Analyst",
  "experience_years": 2
}

TEST 6c – Memory (correction: update only the experience)
🧑 User: Sorry, cor

### Test summary

In [14]:
print(f"{'Test':<70} Result")
print("-" * 80)
for title, passed in test_results:
    print(f"{title:<70} {'✅ PASS' if passed else '❌ FAIL'}")
print("-" * 80)
print(f"{sum(p for _, p in test_results)}/{len(test_results)} tests passed")

Test                                                                   Result
--------------------------------------------------------------------------------
TEST 1 – Conversational                                                ✅ PASS
TEST 2 – Conversational                                                ✅ PASS
TEST 3 – Conversational                                                ✅ PASS
TEST 4 – Extraction (with experience)                                  ✅ PASS
TEST 5 – Extraction (no experience given -> field omitted, not 0)      ✅ PASS
TEST 6a – Memory (partial info: name only, no JSON expected yet)       ❌ FAIL
TEST 6b – Memory (combine with previous turn)                          ❌ FAIL
TEST 6c – Memory (correction: update only the experience)              ❌ FAIL
--------------------------------------------------------------------------------
5/8 tests passed


## 10. Optional Gradio Interface

A simple chat UI built on top of the **same** `chatbot_response` function. Users can ask career questions or type their details, and extractions are shown as a formatted JSON block.

* Each browser session gets its own memory (via Gradio's `session_hash`).
* Clearing the chat also clears that session's memory.
* This section is optional. The chatbot works without it, so you can skip these cells.

In [15]:
!pip install -q gradio

In [16]:
import gradio as gr


def gradio_chat(message: str, history: list, request: gr.Request) -> str:
    session_id = f"gradio-{request.session_hash}" if request else "gradio"
    if not history:  # new or cleared chat -> start with empty memory
        clear_memory(session_id)

    response = chatbot_response(message, session_id)
    # Show extractions as a JSON code block, normal answers as-is
    return f"```json\n{response}\n```" if is_json(response) else response


demo = gr.ChatInterface(
    fn=gradio_chat,
    title="🤝 Recruitment Assistant",
    description="Ask career questions, or tell me your name, the role you want and your experience to get a structured JSON profile.",
    examples=[
        "How do I prepare for a data analyst interview?",
        "How can I make my CV stronger?",
        "My name is Sam and I am applying for a developer role with 3 years of experience.",
    ],
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f47682dfe485c9bb30.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. Reflection

**How function calling improved response consistency.** Just asking the model to "reply in JSON" isn't reliable. It can add a friendly sentence before the JSON, rename keys, or fill in `"experience_years": 0` for someone who never mentioned experience. With the `extract_candidate_info` tool, the model has to fill in a fixed schema, and my Pydantic `CandidateInfo` model validates the result (types, required fields, Title Case roles). Because `experience_years` is `Optional` and I dump the model with `exclude_none=True`, missing experience is left out instead of invented. I also added a small check that drops experience if the user never talked about years at all. The output is always the same keys in the same format, which is what an HR system needs.

**How prompt engineering affected tone and formatting.** The role prompt made a big difference to the tone: answers became warmer and more encouraging. The few-shot examples set the length and layout (one opening line plus 3–5 bullet points), so the three career answers in my tests have a very similar structure. I put the examples inside the system prompt instead of as chat messages so the bot wouldn't think "Sam" from the examples was the real user. The two-turn "Priya" example also showed the model when *not* to call the tool yet, and that is exactly what the John memory test checks.

**One production improvement.** The memory is just a Python dictionary, so everything is lost when the notebook restarts. In a real product I'd store each conversation and the extracted candidate records in a database (e.g. PostgreSQL) and send them to the agency's ATS through its API, with a confirmation step so candidates can check their details before they're submitted.